In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Lesson 1: Getting Started with JAX on GPU

## Overview

JAX is a Python library for high-performance numerical computing and machine learning research. On the surface it looks a lot like NumPy — you write familiar array code — but under the hood JAX traces your Python functions, compiles them with **XLA**, and runs the result on hardware accelerators like NVIDIA GPUs. That combination is what makes JAX fast: you keep writing idiomatic Python, and the compiler turns it into efficient device code.

Getting started on a GPU is often where new users get stuck. CUDA versions, `jaxlib` wheels, container images, environment variables, device visibility — there are many small things that can go wrong before your first array even lands on a GPU. 

**What you'll do:**

* Understand how **JAX**, **XLA**, **CUDA**, and **cuDNN** fit together on an NVIDIA GPU
* Verify that JAX can see your GPU with `jax.devices()`
* Write your first JAX computation and confirm it runs on the GPU
* Compare `jax.numpy` against NumPy to see where JAX feels familiar — and where it differs
* Meet the three transformations that make JAX powerful: **`jax.jit`**, **`jax.grad`**, and **`jax.vmap`**

## Hardware and software requirements

This notebook runs on any machine with at least one modern NVIDIA GPU — H100, B200, or similar. 

> **Driver requirement:** JAX’s pip CUDA wheels include CUDA, cuDNN, and other user-space libraries, but they still require a compatible NVIDIA driver on the host.
> For current JAX wheels, CUDA 13 requires NVIDIA driver **580+** on Linux, while CUDA 12 requires driver **525+**.
> On managed environments, the driver is usually provided by the GPU node image or container runtime.

The table below lists the minimum versions of the rest of the stack.

| Requirement | Minimum                  | Recommended                         |
| ----------- | ------------------------ | ----------------------------------- |
| GPU         | SM 7.5+ (Volta or newer) | SM 9.0+ (Hopper, Blackwell)         |
| CUDA        | 12.x                     | 13.x (latest)                       |
| cuDNN       | 9.x                      | Latest                              |
| JAX         | 0.4.30+                  | Latest                              |
| Python      | 3.10+                    | 3.12                                |

### Two ways to set up JAX on GPU

You have two equally valid paths to a working environment. Pick whichever fits your infrastructure.

**Option 1 — NVIDIA JAX container (recommended).** The container ships JAX, CUDA, cuDNN, NCCL, and several optimized libraries together as a tested combination. This is the fastest way to avoid version-mismatch issues, and it's how this course is meant to be run.

Browse and pull the latest tag from the NGC catalog: [NVIDIA JAX container on NGC](https://catalog.ngc.nvidia.com/orgs/nvidia/containers/jax).

A typical launch looks like:

```bash
docker run --rm -it --gpus=all --shm-size=16g \
  -v "$PWD":/workspace -w /workspace \
  nvcr.io/nvidia/jax:<tag> \
  bash
```

Replace `<tag>` with the current tag listed on the NGC page.

**Option 2 — pip install.** If you can't (or don't want to) use the container, install JAX with CUDA support directly from PyPI. We recommend the **latest CUDA 13 build** — it has the best performance on Hopper and Blackwell GPUs, and the `cuda13` extra pulls in matching CUDA and cuDNN wheels alongside `jaxlib`, so you don't need a system CUDA toolkit:

```bash
pip install --upgrade "jax[cuda13]"
```

If you're stuck on CUDA 12 for compatibility reasons, the equivalent `cuda12` extra still works:

```bash
pip install --upgrade "jax[cuda12]"
```

Both extras work in a plain Python virtual environment, a Conda environment, or a generic CUDA base image. You only need a recent NVIDIA driver installed on the host; the CUDA libraries themselves come from the wheels.

> **Already have CUDA installed?** The `cuda13-local` and `cuda12-local` extras skip the bundled CUDA binaries and link against your system install instead — handy for HPC clusters with site-managed CUDA. The official JAX [installation guide](https://docs.jax.dev/en/latest/installation.html) lists every supported combination.

Either path leads to the same place: a Python process where `import jax` works and `jax.devices()` returns a GPU.

## How JAX runs on a GPU

JAX itself is a Python library. It does **not** run kernels on the GPU directly — instead, it builds a *program* describing your computation and hands that program to a stack of lower-level components. For an NVIDIA GPU, that stack looks like this:

> **The JAX-on-GPU stack**
> - **JAX (Python)** — traces your Python function into an intermediate representation (`jaxpr` / StableHLO)
> - **XLA** — Google's compiler that turns the IR into optimized GPU code
> - **cuDNN, cuBLAS, NCCL** — NVIDIA libraries that XLA calls into for common primitives (convolutions, GEMMs, collectives)
> - **CUDA driver and runtime** — the foundation that loads kernels onto the GPU and manages device memory

The key idea: when you call a JAX function, JAX records which operations you performed on which shapes and dtypes, hands the result to XLA, and XLA produces a single optimized kernel (or sequence of kernels) for your GPU. You almost never write CUDA yourself — but the speed you get is comparable to hand-written CUDA for most workloads.

## Setup

First, let's confirm we have a working GPU and a working JAX install. The `nvidia-smi` command shows the GPU model, driver version, and current memory usage — if this doesn't return useful output, nothing downstream will work either.

In [ ]:
!nvidia-smi

We can also ask `nvidia-smi` for the GPU's **compute capability** — a two-digit number that identifies the hardware generation (8.0 = Ampere, 9.0 = Hopper, 10.0 = Blackwell). Most JAX features work on SM 7.0+, but optimized attention and FP8 paths need newer hardware.

In [ ]:
import subprocess


def get_compute_capability() -> tuple[int, int]:
    """Query the compute capability of the first visible GPU."""
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
        text=True,
    )
    major, minor = out.strip().split("\n")[0].split(".")
    return int(major), int(minor)


SM_MAJOR, SM_MINOR = get_compute_capability()
print(f"Detected compute capability: SM {SM_MAJOR}.{SM_MINOR}")

if SM_MAJOR < 7:
    print("WARNING: this course assumes SM 7.0+ (Volta or newer).")
else:
    print("GPU is compatible with this course.")

Now let's check that JAX is installed and can see the GPU. The first import takes a few seconds because JAX initializes the CUDA runtime and probes the available devices.

In [ ]:
import jax
import jax.numpy as jnp

devices = jax.devices()
gpu_devices = [d for d in devices if d.platform == "gpu"]

print(f"JAX version:        {jax.__version__}")
print(f"Default backend:    {jax.default_backend()}")
print(f"Available devices:  {devices}")

assert gpu_devices, f"No GPU backend found. Available devices: {devices}"
print(f"GPU devices:        {gpu_devices}")

You should see one or more entries of the form `CudaDevice(id=0)`, and the default backend should report `'gpu'`. If `jax.devices()` returns a `CpuDevice` instead, JAX did not find a working GPU backend — usually because of a mismatched CUDA version, a missing driver, or `jaxlib` installed without the CUDA extra. Re-check the setup options above: the NGC container ships everything pre-configured, and `pip install "jax[cuda13]"` brings its own CUDA/cuDNN wheels.

> **Tip:** If you ever want to force JAX onto the CPU (for a quick debugging comparison, say), set `JAX_PLATFORMS=cpu` in the environment *before* importing JAX. The default is to use the GPU when one is available.

A few **quick checks** usually catch the common issues:

* Make sure the host NVIDIA driver is new enough for your CUDA path.
* Restart the notebook kernel after changing `jax`, `jaxlib`, or CUDA-related packages.
* If you installed JAX with bundled CUDA wheels, unset `LD_LIBRARY_PATH` unless you know you need it;
  it can cause JAX to pick up incompatible system CUDA libraries.
* Set environment variables such as `JAX_PLATFORMS`, `CUDA_VISIBLE_DEVICES`, or
  `XLA_PYTHON_CLIENT_MEM_FRACTION` before importing JAX.

## JAX feels like NumPy

The fastest way to get comfortable with JAX is to notice that `jax.numpy` feels a lot like NumPy. Many of the same array constructors and broadcasting rules work as you would expect from NumPy, but JAX has a few important differences. In particular, JAX defaults to 32-bit dtypes unless 64-bit mode is enabled with `jax_enable_x64`, which is better suited to accelerator-heavy AI workloads. The other big difference is *where* the array lives and *how* the computation runs.

Let's compute the same expression two ways — with NumPy and with JAX — and check where each result is stored.

In [ ]:
import numpy as np

# NumPy: runs on the CPU, stored in host memory
x_np = np.arange(8, dtype=np.float32)
y_np = np.sin(x_np) ** 2 + np.cos(x_np) ** 2
print(f"NumPy result:  {y_np}")
print("NumPy device:  CPU (host memory)")
print()

# JAX: same code, different array library
x = jnp.arange(8, dtype=jnp.float32)
y = jnp.sin(x) ** 2 + jnp.cos(x) ** 2
print(f"JAX result:    {y}")
print(f"JAX device:    {y.device}")

# Sanity check: the two answers should agree
np.testing.assert_allclose(y_np, np.asarray(y), atol=1e-6)
print()
print("NumPy and JAX agree.")

Notice three things:

1. The code is identical apart from `np` vs `jnp`.
2. `y.device` reports a CUDA device — JAX placed the array on the GPU automatically because that is the default backend.
3. JAX returns its own array type (`jax.Array`), not a NumPy array. If you need a NumPy view, call `np.asarray(y)` — which triggers a **GPU → CPU transfer**. Printing a JAX array also forces synchronization, because Python has to fetch the value in order to display it. That is fine for tiny examples like this, but avoid printing inside timed GPU code.

That last point matters more than it looks. Every time you cross between GPU and host, you pay a cost. A good mental rule for JAX-on-GPU code is *create your arrays with `jnp`, operate on them with `jnp`, and only convert to NumPy when you actually need to look at the values*. We'll see in next lessons how to spot accidental host transfers in a profile.

> **NumPy-like, not identical:** JAX arrays are immutable, so in-place updates like `x[i] = y`
> are not supported. Use functional updates such as `x = x.at[i].set(y)` instead. JAX also defaults
> to 32-bit dtypes unless 64-bit mode is enabled, which is different from NumPy’s usual defaults.

## The three transformations that make JAX *JAX*

NumPy-on-GPU is nice, but on its own it isn't a giant leap over alternatives like [CuPy](https://cupy.dev/). What makes
JAX distinctive is its **function transformations**: small operators that take a Python function
and return a *new* Python function with extra powers.

Three of them are essential, and you'll use them in every lesson going forward:

* **`jax.jit`** — trace and compile a function so it runs as a single fused GPU program
* **`jax.grad`** — return a new function that computes the gradient of the original
* **`jax.vmap`** — vectorize a function across a batch dimension, with no Python loop

Let's see each of them in action on tiny examples.

### `jax.jit`: compile your function

When you call a plain JAX function, operations are dispatched to the GPU as they execute. That works, but each dispatch has overhead, and small operations can leave the GPU underused. **`jax.jit`** changes the execution model: JAX traces your function, XLA compiles it into an optimized executable for the target device, and JAX reuses that compiled executable on later calls with compatible input shapes and dtypes. XLA may fuse many operations together, but a compiled function can still lower to multiple GPU kernels.

The first call is slow — that's the compile. Every call after is fast.

In [ ]:
import time


def f(x):
    """Compose tanh, sin, and log1p so XLA has multiple ops to fuse when jitted."""
    # A few cheap ops chained together so compilation has something to fuse
    return jnp.tanh(x) * jnp.sin(x) + jnp.log1p(x * x)


x = jnp.arange(1_000_000, dtype=jnp.float32)

# Eager: one kernel launch per operation
_ = f(x).block_until_ready()  # warm up
t0 = time.perf_counter()
for _ in range(10):
    y = f(x).block_until_ready()
eager_ms = (time.perf_counter() - t0) * 1000 / 10
print(f"Eager:            {eager_ms:6.3f} ms / call")

# Compiled: optimized executable, often with fused operations
f_jit = jax.jit(f)
_ = f_jit(x).block_until_ready()  # first call compiles
t0 = time.perf_counter()
for _ in range(10):
    y = f_jit(x).block_until_ready()
jit_ms = (time.perf_counter() - t0) * 1000 / 10
print(f"jax.jit (cached): {jit_ms:6.3f} ms / call")
print(f"Speedup:          {eager_ms / jit_ms:6.1f}x")

The exact speedup depends on the size and shape of your computation, but the pattern is
universal: **eager JAX is convenient, compiled JAX is fast**. In the next lessons we'll dive into how `jit` traces and compiles your code, and what causes the dreaded "why is JAX recompiling again?" slowdown.

> **Heads up:** `block_until_ready()` is essential for honest timing. JAX dispatches work to the
> GPU asynchronously, so without the block you'd be measuring how long it took to *schedule* the
> work, not how long it took to run.

### `jax.grad`: automatic differentiation

Training a neural network means computing gradients of a loss with respect to parameters. JAX
gives you those gradients automatically: pass any scalar-valued function to **`jax.grad`**, and it
returns a new function that computes the derivative.

Let's differentiate a tiny mean-squared-error loss with respect to a single parameter `w`.

In [ ]:
def loss(w, x, y):
    """Mean squared error of `w*x` vs `y`; scalar loss for the `jax.grad` demo below."""
    pred = w * x
    return jnp.mean((pred - y) ** 2)


w = jnp.array(0.5)
xs = jnp.array([1.0, 2.0, 3.0, 4.0])
ys = jnp.array([2.0, 4.0, 6.0, 8.0])  # true relationship: y = 2x

# grad returns a function with the same signature, differentiating w.r.t. the first argument
dloss_dw = jax.grad(loss)

print(f"loss(w=0.5):   {loss(w, xs, ys):.4f}")
print(f"dloss/dw:      {dloss_dw(w, xs, ys):.4f}")

# Sanity check against a finite-difference approximation
eps = 1e-3
fd = (loss(w + eps, xs, ys) - loss(w - eps, xs, ys)) / (2 * eps)
print(f"finite diff:   {fd:.4f}  (should match)")

The gradient is negative, which tells an optimizer that increasing `w` will decrease the
loss — exactly what we'd expect, since the true relationship is `y = 2x` and we started at
`w = 0.5`. We'll build a full training loop around `jax.grad` in Lesson 4.

### `jax.vmap`: automatic vectorization

GPUs love batched work. The naive way to apply a function to many inputs is a Python `for` loop,
but that launches kernels one at a time and starves the GPU. The traditional fix is to rewrite the
function in terms of batch dimensions — which is tedious and error-prone.

**`jax.vmap`** takes a function written for *a single example* and returns a version that operates
on *a batch*, with no loop and no manual reshaping.

In [ ]:
def predict(W, x):
    """Tanh of a single-example matrix-vector product; vmapped below to batch over many `x`."""
    # Single example: W is (out, in), x is (in,) -> result is (out,)
    return jnp.tanh(W @ x)


key_w, key_x = jax.random.split(jax.random.key(0))
W = jax.random.normal(key_w, (4, 3))
xs = jax.random.normal(key_x, (10, 3))  # batch of 10 examples

# Without vmap: a Python loop, one kernel launch per example
ys_loop = jnp.stack([predict(W, x) for x in xs])

# With vmap: batch over the leading axis of xs, share W across the batch
batched_predict = jax.vmap(predict, in_axes=(None, 0))
ys_vmap = batched_predict(W, xs)

print(f"ys_loop shape:  {ys_loop.shape}")
print(f"ys_vmap shape:  {ys_vmap.shape}")
np.testing.assert_allclose(np.asarray(ys_loop), np.asarray(ys_vmap), atol=1e-6)
print("vmap matches the explicit loop.")

The `in_axes=(None, 0)` argument says: *don't batch `W` (broadcast it), do batch `xs` along axis 0*. The result is identical to the loop, but it dispatches as a single batched GPU operation.

The real superpower is **composition**. You can stack transformations:

```python
fast_batched_grad = jax.jit(
    jax.vmap(jax.grad(loss), in_axes=(None, 0, 0))
)
```

That one line gives you a compiled, vectorized, differentiated function — most of what you need for batched training. 

## Summary

In this first lesson, you set up the foundation for the rest of the course: a working mental model of how JAX uses NVIDIA GPUs, plus a few quick checks that tell you whether your environment is actually using the GPU.

In this notebook you learned to:

* **Recognize** the JAX-on-GPU stack: JAX traces, XLA compiles, and CUDA libraries such as cuDNN and cuBLAS execute work on the GPU.
* **Verify** that JAX sees your GPU with `jax.devices()` and `jax.default_backend()`.
* **Write** NumPy-like array code with `jax.numpy` and confirm the result lives on the GPU.
* **Understand** where JAX differs from NumPy, including immutable arrays, `.at[...]` updates, and default 32-bit dtypes.
* **Apply** `jax.jit` to compile a function and compare compiled execution with eager execution.
* **Differentiate** scalar functions automatically with `jax.grad`.
* **Vectorize** single-example functions over a batch with `jax.vmap`.

If every cell above ran cleanly, your environment is correctly set up for the rest of the course.